In [ ]:
# 05. 일반화 점검: plate_defect_count 효과 + 생산 조건 그룹 교차검증
import sys, warnings
sys.path.insert(0, "../src"); warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from steel import *
pd.set_option("display.width", 200); pd.set_option("display.precision", 4)
from joblib import Parallel, delayed
from sklearn.metrics import f1_score
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedGroupKFold
from xgboost import XGBClassifier

X_train, X_test, y_train, y_test = load_data()
XF = add_features(X_train)
groups = plate_groups(X_train).values

In [ ]:
# plate_defect_count를 뺀 35개 피처로 구성 모델 3개 재튜닝 (03과 같은 탐색 공간·횟수·난수)
SP35 = search_spaces(XF.columns, use_count=False, models=("ExtraTrees", "RandomForest", "GBM"))
T35 = {n: cached(f"tune35_{n}", lambda n=n: tune(n, SP35[n], XF, y_train)) for n in SP35}
P35 = {n: r["params"] for n, r in T35.items()}
pd.DataFrame({n: {"35개 재튜닝 macro_f1": r["score"]} for n, r in T35.items()}).T

In [ ]:
# 비교 모델 (모두 같은 분할에서 평가)
MODELS = {
    "Baseline XGBoost (26)": (lambda: make_pipeline(LabelEncodedClassifier(XGBClassifier(random_state=RS, n_jobs=1)), X_train.columns), X_train),
    "Final SV (36, count 포함)": (lambda: final_model(XF.columns), XF),
    "SV (35, count 제외)": (lambda: final_model(XF.columns, params=P35, use_count=False), XF),
}
SPLITS = {
    "random 5×3": list(RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RS).split(XF, y_train)),
    "group 5×3": [s for r in range(3) for s in StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RS + r).split(XF, y_train, groups)],
}
assert all(not set(groups[tr]) & set(groups[va]) for tr, va in SPLITS["group 5×3"])   # 그룹 분할: 학습·검증 조건 겹침 없음

def fold(name, cv, i):
    make, X = MODELS[name]; tr, va = SPLITS[cv][i]
    return name, cv, i, make().fit(X.iloc[tr], y_train.iloc[tr]).predict(X.iloc[va])
jobs = [(n, cv, i) for n in MODELS for cv in SPLITS for i in range(15)]
PRED = cached("generalization_pred", lambda: {(n, cv, i): p for n, cv, i, p in Parallel(n_jobs=-1)(delayed(fold)(*j) for j in jobs)})
F1 = {(n, cv): np.array([f1_score(y_train.iloc[SPLITS[cv][i][1]], PRED[(n, cv, i)], average="macro") for i in range(15)]) for n in MODELS for cv in SPLITS}

In [ ]:
# 무작위 분할(이미 본 생산 조건) vs 그룹 분할(처음 보는 생산 조건), 15 fold 평균
res = pd.DataFrame({cv: {n: F1[(n, cv)].mean() for n in MODELS} for cv in SPLITS})
res["drop"] = res["group 5×3"] - res["random 5×3"]
res
# → 처음 보는 생산 조건에서는 모든 모델이 크게 하락 (일반 CV는 새 조건의 성능을 부풀림)

In [ ]:
# plate_defect_count 포함 vs 제외: 같은 fold끼리 짝 비교
inc, exc = "Final SV (36, count 포함)", "SV (35, count 제외)"
pd.DataFrame({cv: {"포함 - 제외 (평균)": (F1[(inc, cv)] - F1[(exc, cv)]).mean(), "포함이 좋은 fold": f"{(F1[(inc, cv)] > F1[(exc, cv)]).sum()}/15"} for cv in SPLITS}).T
# → 이 변수는 이미 본 조건에서는 조건의 이름표처럼 도움이 되지만, 새 조건에서는 항상 0이 되어 오히려 불리할 수 있음

In [ ]:
# 그룹 분할에서 결함 유형별 F1 (최종 모델, 15 fold 예측 합산)
def per_class(name, cv):
    yt = np.concatenate([y_train.values[SPLITS[cv][i][1]] for i in range(15)])
    yp = np.concatenate([PRED[(name, cv, i)] for i in range(15)])
    return f1_score(yt, yp, labels=CLASS_ORDER, average=None)
pc = pd.DataFrame({cv: per_class(inc, cv) for cv in SPLITS}, index=CLASS_ORDER)
pc["drop"] = pc["group 5×3"] - pc["random 5×3"]
pc
# → 생산 조건에 크게 기대던 유형일수록 새 조건에서 크게 하락